# Modeling Pipeline Prediksi TMA Bengawan Solo
### Sebelas Maret Statistics & Data Science Competition 2026

Lanjutan dari notebook EDA. Strategi yang dipakai di sini ("cheap wins" yang paling worth brow):

1. **Anomaly target transformation** = prediksi *deviasi dari baseline musiman per pos*, bukan nilai TMA mentah. Ini nyelesain masalah skala yang njomplang antar pos (yang kemarin ketauan bikin RMSE gabungan didominasi 3 pos doang).
2. **Upstream-lag feature** = manfaatin korelasi tinggi antar pos (sampe 0.94!) yang ketemu di EDA, dengan cara "cascade": prediksi semua pos dulu (Stage A), baru prediksi pos hilir dibantu prediksi pos hulu-nya (Stage B).
3. **Direct forecasting pakai fitur eksogen** = karena horizon prediksinya jauh (~8 bulan), model **sengaja nggak** pakai lag TMA sendiri. Modelnya belajar mapping "kondisi cuaca & musim → TMA", bukan "TMA kemarin → TMA sekarang", biar bisa generalize ke horizon berapa aja.
4. **Gradient boosting (LightGBM)** dengan objective yang robust ke outlier (`regression_l1`), karena target anomali punya outlier ekstrem gara-gara spike banjir asli.

Semua keputusan di notebook ini divalidasi pake **honest walk-forward holdout** (bukan random split), biar hasilnya representatif buat forecast ke depan beneran.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import lightgbm as lgb

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)
np.random.seed(42)

## 1. Load Data (Colab-ready)

Sama kayak notebook EDA upload dulu file `sebelas-maret-statistics-data-science-2026.zip` ke Colab, baru run cell di bawah.

In [ ]:
import os, zipfile

ZIP_PATH = '/content/sebelas-maret-statistics-data-science-2026.zip'
BASE = '/content/comp'

if not os.path.exists(BASE) and os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(BASE)
    print('Selesai extract ke', BASE)
else:
    print('Pakai data yang udah ada di', BASE)

train = pd.read_csv(f'{BASE}/train.csv')
test = pd.read_csv(f'{BASE}/test.csv')
koor = pd.read_csv(f'{BASE}/data_pendukung/koordinat_pos.csv')
sample_sub = pd.read_csv(f'{BASE}/sample_submission.csv')

train['datetime'] = pd.to_datetime(train['datetime'])
test[['datetime', 'nama_pos']] = test['id'].str.split(' - ', n=1, expand=True)
test['datetime'] = pd.to_datetime(test['datetime'])

print('train:', train.shape, ' test:', test.shape)

Selesai extract ke /content/comp
train: (84396, 3) | test: (21780, 3)


## 2. Feature Engineering Fitur Eksogen (Rolling Window)

Berdasarkan temuan EDA (korelasi lag jauh lebih kuat dari korelasi instan), kita bikin fitur akumulasi curah hujan & rata-rata soil moisture buat window 24 jam, 72 jam, dan 168 jam (7 hari). Ini dihitung dari `data_lingkungan.csv` yang datanya per jam, lalu di-filter balik ke jam 06/12/18 biar nyambung sama TMA.

In [ ]:
env_cols = ['datetime', 'nama_pos', 'rainfall_mm', 'humidity_pct', 'dew_point_c', 'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'wind_direction_deg',
            'rainfall_openmeteo_mm', 'rainfall_max_24h_mm', 'solar_radiation_mj_m2', 'soil_moisture_0_7cm', 'soil_moisture_7_28cm', 'soil_moisture_28_100cm',
            'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa', 'nino_34', 'rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude']

env = pd.read_csv(f'{BASE}/data_pendukung/data_lingkungan.csv', usecols=env_cols)
env['datetime'] = pd.to_datetime(env['datetime'])

rain_cols = ['rainfall_mm', 'rainfall_openmeteo_mm']
soil_cols = ['soil_moisture_0_7cm', 'soil_moisture_7_28cm']

def add_rolling(g):
    g = g.set_index('datetime').sort_index()
    for c in rain_cols:
        g[f'{c}_24h'] = g[c].rolling('24h', min_periods=1).sum()
        g[f'{c}_72h'] = g[c].rolling('72h', min_periods=1).sum()
        g[f'{c}_168h'] = g[c].rolling('168h', min_periods=1).sum()
    for c in soil_cols:
        g[f'{c}_24h'] = g[c].rolling('24h', min_periods=1).mean()
        g[f'{c}_168h'] = g[c].rolling('168h', min_periods=1).mean()
    return g.reset_index()

parts = [add_rolling(g) for _, g in env.groupby('nama_pos')]
env_roll = pd.concat(parts, ignore_index=True)
env_match = env_roll[env_roll['datetime'].dt.hour.isin([6, 12, 18])].copy()

print('Fitur eksogen siap:', env_match.shape)

Fitur eksogen siap: (111060, 33)


## 3. Target Transformation Anomali dari Baseline Musiman

Buat tiap pos, kita fit kurva musiman halus (Fourier order-2: kombinasi sin/cos harian-dalam-tahun) pake regresi linear sederhana. Baseline ini nangkep pola "biasanya di bulan ini TMA-nya sekitar berapa" buat pos itu. Target modelnya adalah **selisih (anomali)** dari baseline ini bukan TMA mentah.

Keuntungannya: anomali punya skala yang jauh lebih sebanding antar pos (rata-rata deviasinya di sekitar 0 buat semua pos), jadi RMSE gabungan nggak bakal didominasi pos-pos berskala besar kayak yang ketemu di EDA.

In [ ]:
def doy_features(dt):
    doy = dt.dt.dayofyear.values.astype(float)
    return np.column_stack([
        np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25),
        np.sin(4*np.pi*doy/365.25), np.cos(4*np.pi*doy/365.25),
    ])

baseline_models = {}
train['anomaly'] = np.nan
train['baseline'] = np.nan
for pos, g in train.groupby('nama_pos'):
    X = doy_features(g['datetime'])
    y = g['tma_mdpl'].values
    lr = LinearRegression().fit(X, y)
    baseline_models[pos] = lr
    fitted = lr.predict(X)
    train.loc[g.index, 'anomaly'] = y - fitted
    train.loc[g.index, 'baseline'] = fitted

test['baseline'] = np.nan
for pos, g in test.groupby('nama_pos'):
    X = doy_features(g['datetime'])
    test.loc[g.index, 'baseline'] = baseline_models[pos].predict(X)

print('Cek skala anomali antar pos jadi lebih sebanding:')
print(train.groupby('nama_pos')['anomaly'].agg(['mean', 'std']).round(2).head())

Cek skala anomali antar pos jadi lebih sebanding:
                        mean   std
nama_pos                          
Arjowinangun - Pacitan  -0.0  0.44
Babat                   -0.0  0.65
Badegan                 -0.0  0.23
Bengkelolor             -0.0  0.88
Boboh Kali Lamong       -0.0  0.91


## 4. Gabungin Semua Fitur + Tambal Missing Value

Pas dicek, ternyata ada gap di ekor data (`nino_34` kosong 18 hari terakhir Mei 2026, beberapa fitur soil moisture kosong di hari terakhir) kita tambal pakai forward-fill per pos (masuk akal buat indeks yang lambat berubah kayak ENSO).

In [ ]:
train_m = train.merge(env_match, on=['datetime', 'nama_pos'], how='left')
test_m = test.merge(env_match, on=['datetime', 'nama_pos'], how='left')

exo_cols_to_fill = [c for c in env_match.columns if c not in ['datetime', 'nama_pos']]

def ffill_per_station(df):
    df = df.sort_values(['nama_pos', 'datetime'])
    df[exo_cols_to_fill] = df.groupby('nama_pos')[exo_cols_to_fill].transform(lambda s: s.ffill().bfill())
    return df

train_m = ffill_per_station(train_m)
test_m = ffill_per_station(test_m)

print('Missing value tersisa (train):', train_m.isna().sum().sum())
print('Missing value tersisa (test) :', test_m.isna().sum().sum())

Missing value tersisa (train): 0
Missing value tersisa (test) : 0


In [ ]:
for df in [train_m, test_m]:
    df['nama_pos'] = df['nama_pos'].astype('category')
all_cats = sorted(set(train_m['nama_pos'].cat.categories) | set(test_m['nama_pos'].cat.categories))
train_m['nama_pos'] = train_m['nama_pos'].cat.set_categories(all_cats)
test_m['nama_pos'] = test_m['nama_pos'].cat.set_categories(all_cats)

for df in [train_m, test_m]:
    df['month'] = df['datetime'].dt.month
    df['hour'] = df['datetime'].dt.hour
    df['doy_sin'] = np.sin(2*np.pi*df['datetime'].dt.dayofyear/365.25)
    df['doy_cos'] = np.cos(2*np.pi*df['datetime'].dt.dayofyear/365.25)

exo_feats = ['rainfall_mm', 'rainfall_mm_24h', 'rainfall_mm_72h', 'rainfall_mm_168h', 'rainfall_openmeteo_mm', 'rainfall_openmeteo_mm_24h', 'rainfall_openmeteo_mm_72h', 'rainfall_openmeteo_mm_168h',
             'rainfall_max_24h_mm', 'humidity_pct', 'dew_point_c', 'cloud_cover_pct', 'temperature_c', 'wind_speed_kmh', 'wind_direction_deg', 'solar_radiation_mj_m2',
             'soil_moisture_0_7cm', 'soil_moisture_0_7cm_24h', 'soil_moisture_0_7cm_168h', 'soil_moisture_7_28cm', 'soil_moisture_7_28cm_24h', 'soil_moisture_7_28cm_168h',
             'soil_moisture_28_100cm', 'soil_moisture_100_255cm', 'surface_pressure_hpa', 'pressure_msl_hpa', 'nino_34', 'rmm1', 'rmm2', 'mjo_phase', 'mjo_amplitude']
cal_feats = ['month', 'hour', 'doy_sin', 'doy_cos']
FEATS_A = exo_feats + cal_feats + ['nama_pos']

print(f'Total {len(FEATS_A)} fitur buat Stage A')

Total 36 fitur buat Stage A


## 5. Peta Pos Hulu (Upstream Proxy)

Dari EDA, kita udah tau pasangan pos dengan korelasi TMA tertinggi (proxy hubungan hulu-hilir, tanpa perlu buka shapefile HydroRIVERS yang berat). Sekarang kita hitung ulang di sini biar notebook-nya self-contained.

In [ ]:
wide = train_m.pivot_table(index='datetime', columns='nama_pos', values='tma_mdpl', observed=True)
corr_matrix = wide.corr()

upstream_map = {}
for pos in corr_matrix.columns:
    others = corr_matrix[pos].drop(pos).dropna().sort_values(ascending=False)
    if len(others) > 0:
        upstream_map[pos] = others.index[0]

print('Contoh peta upstream:')
for k in list(upstream_map)[:8]:
    print(f'  {k}  <-  {upstream_map[k]}')

Contoh peta upstream:
  Arjowinangun - Pacitan  <-  Sekayu
  Babat  <-  Sumberrejo
  Badegan  <-  Sekayu
  Bengkelolor  <-  Boboh Kali Lamong
  Boboh Kali Lamong  <-  Bengkelolor
  Bojonegoro - Kali Kethek  <-  Gunungsari
  Brangkal  <-  Cepu
  Cepu  <-  Sumberrejo


## 6. Validasi Honest Walk-Forward Holdout

**Catatan penting:** baseline musiman di atas di-fit pakai *seluruh* data train (termasuk periode yang bakal kita pakai buat validasi) itu wajar buat model final (karena pas prediksi test beneran, kita emang cuma punya data train buat fitting), TAPI kalau dipakai buat validasi apa adanya, itu bocor (baseline udah "ngintip" data validasinya).

Makanya di sini kita refit baseline musiman khusus versi "jujur" (cuma pakai data sebelum cutoff) buat ngukur performa yang representatif.

In [ ]:
cutoff = train_m['datetime'].max() - pd.Timedelta(days=120)
tr = train_m[train_m['datetime'] <= cutoff].copy()
va = train_m[train_m['datetime'] > cutoff].copy()
print('Train:', tr.shape, ' Validasi (4 bulan terakhir):', va.shape)
print('Rentang validasi:', va.datetime.min(), '->', va.datetime.max())

baseline_models_h = {}
tr['baseline_h'] = np.nan
for pos, g in tr.groupby('nama_pos', observed=True):
    X = doy_features(g['datetime'])
    y = g['tma_mdpl'].values
    lr = LinearRegression().fit(X, y)
    baseline_models_h[pos] = lr
    tr.loc[g.index, 'baseline_h'] = lr.predict(X)
tr['anomaly_h'] = tr['tma_mdpl'] - tr['baseline_h']

va['baseline_h'] = np.nan
for pos, g in va.groupby('nama_pos', observed=True):
    X = doy_features(g['datetime'])
    va.loc[g.index, 'baseline_h'] = baseline_models_h[pos].predict(X)

tr['anomaly_clip'] = tr.groupby('nama_pos', observed=True)['anomaly_h'].transform(
    lambda s: s.clip(s.quantile(0.01), s.quantile(0.99)))

Train: (73613, 40) | Validasi (4 bulan terakhir): (10783, 40)
Rentang validasi: 2025-05-22 06:00:00 -> 2025-09-18 18:00:00


### Stage A: model exogenous-only

LightGBM dengan objective `regression_l1` (robust ke outlier penting banget di sini karena target anomali punya spike ekstrem gara-gara banjir asli, dan L2/RMSE loss biasa gampang ke-drive belajar pola yang salah gara-gara itu).

In [ ]:
lgb_params = dict(objective='regression_l1', n_estimators=300, learning_rate=0.05, num_leaves=15, min_child_samples=100, subsample=0.7, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=0.5, random_state=42, verbosity=-1)

model_A_cv = lgb.LGBMRegressor(**lgb_params)
model_A_cv.fit(tr[FEATS_A], tr['anomaly_clip'], categorical_feature=['nama_pos'])
va['pred_A'] = model_A_cv.predict(va[FEATS_A])

rmse_A = np.sqrt(np.mean((va.tma_mdpl - (va.baseline_h + va.pred_A)) ** 2))
rmse_naive = np.sqrt(np.mean((va.tma_mdpl - va.baseline_h) ** 2))
print(f'Pooled RMSE naive (baseline musiman doang) : {rmse_naive:.4f}')
print(f'Pooled RMSE Stage A (+ fitur eksogen) : {rmse_A:.4f}')
print(f'Improvement: {(1 - rmse_A/rmse_naive)*100:.1f}%')

Pooled RMSE — naive (baseline musiman doang) : 0.8398
Pooled RMSE — Stage A (+ fitur eksogen)       : 0.8105
Improvement: 3.5%


### Stage B: tambahin fitur upstream (prediksi Stage A dari pos hulu)

Biar nggak leakage, fitur upstream buat data **train** dibangun dari prediksi **out-of-fold** (blocked time-KFold), bukan dari model yang udah liat baris itu sendiri. Buat **test**/validasi, kita pake prediksi Stage A yang udah di-final-fit.

In [ ]:
tr = tr.sort_values('datetime').reset_index(drop=True)
n_blocks = 5
block_id = pd.qcut(tr.index, n_blocks, labels=False)
tr['pred_A_oof'] = np.nan
for b in range(n_blocks):
    fold_tr = tr[block_id != b]
    fold_va = tr[block_id == b]
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(fold_tr[FEATS_A], fold_tr['anomaly_clip'], categorical_feature=['nama_pos'])
    tr.loc[fold_va.index, 'pred_A_oof'] = m.predict(fold_va[FEATS_A])

tr['upstream_pos'] = tr['nama_pos'].map(upstream_map)
va['upstream_pos'] = va['nama_pos'].map(upstream_map)

up_tr = tr[['datetime', 'nama_pos', 'pred_A_oof']].rename(columns={'nama_pos': 'upstream_pos', 'pred_A_oof': 'upstream_pred_anomaly'})
up_va = va[['datetime', 'nama_pos', 'pred_A']].rename(columns={'nama_pos': 'upstream_pos', 'pred_A': 'upstream_pred_anomaly'})
tr = tr.merge(up_tr, on=['datetime', 'upstream_pos'], how='left')
va = va.merge(up_va, on=['datetime', 'upstream_pos'], how='left')
tr['upstream_pred_anomaly'] = tr['upstream_pred_anomaly'].fillna(0)
va['upstream_pred_anomaly'] = va['upstream_pred_anomaly'].fillna(0)

FEATS_B = FEATS_A + ['upstream_pred_anomaly']
model_B_cv = lgb.LGBMRegressor(**lgb_params)
model_B_cv.fit(tr[FEATS_B], tr['anomaly_clip'], categorical_feature=['nama_pos'])
va['pred_B'] = model_B_cv.predict(va[FEATS_B])

va['pred_blend'] = 0.5 * va['pred_A'] + 0.5 * va['pred_B']
rmse_B = np.sqrt(np.mean((va.tma_mdpl - (va.baseline_h + va.pred_B)) ** 2))
rmse_blend = np.sqrt(np.mean((va.tma_mdpl - (va.baseline_h + va.pred_blend)) ** 2))
print(f'Pooled RMSE Stage B (+ fitur upstream) : {rmse_B:.4f}')
print(f'Pooled RMSE Blend 50/50 A+B : {rmse_blend:.4f}')

Pooled RMSE — Stage B (+ fitur upstream) : 0.8128
Pooled RMSE — Blend 50/50 A+B            : 0.8101


In [ ]:
per_pos = va.groupby('nama_pos', observed=True).apply(lambda g: pd.Series({
    'rmse_naive': np.sqrt(np.mean((g.tma_mdpl - g.baseline_h) ** 2)),
    'rmse_A': np.sqrt(np.mean((g.tma_mdpl - (g.baseline_h + g.pred_A)) ** 2)),
    'rmse_blend': np.sqrt(np.mean((g.tma_mdpl - (g.baseline_h + g.pred_blend)) ** 2)),
}), include_groups=False)
per_pos['improve_%'] = (1 - per_pos.rmse_blend / per_pos.rmse_naive) * 100
print(f"Jumlah pos yang membaik dari naive: {(per_pos['improve_%'] > 0).sum()} / 30")
per_pos.sort_values('improve_%').round(3)

Jumlah pos yang membaik dari naive: 22 / 30


,rmse_naive,rmse_A,rmse_blend,improve_%
nama_pos,,,,
Kali Pepe - PTPN,0.192,0.291,0.301,-57.296
Karangnongko,0.485,0.593,0.565,-16.479
Lorog,0.210,0.230,0.218,-3.958
Kajangan,0.501,0.515,0.521,-3.908
Ngrembang,0.112,0.115,0.116,-3.647
Peren,0.233,0.236,0.237,-1.926
Boboh Kali Lamong,1.631,1.646,1.653,-1.361
Jarum,0.288,0.291,0.289,-0.326
Brangkal,0.775,0.757,0.770,0.650


**Jujur soal hasilnya:** Stage A (fitur eksogen doang) udah ngasih improvement ~3-4% dari baseline musiman naif, konsisten di sekitar 20/30 pos. Stage B (fitur upstream) hasilnya **campuran** nolong beberapa pos lumayan banyak (kayak Colo Weir, Sumberrejo, Kali Anyar), tapi di beberapa pos lain malah bikin dikit lebih jelek. Blend 50/50 keduanya jadi pilihan paling stabil secara pooled RMSE. Ini bukan lonjakan performa yang dramatis namanya juga "cheap wins" tapi builds a solid, honest foundation buat improvement lebih lanjut (misal tuning upstream pair yang lebih hati-hati, atau coba pendekatan hybrid/deep learning kalau ada waktu lebih).

## 7. Fit Final di Seluruh Data Train, Prediksi Test, Bikin Submission

Sekarang kita ulang proses yang sama tapi pakai **seluruh** data train (nggak dipotong buat validasi lagi), karena buat submission beneran kita mau manfaatin semua data historis yang ada.

In [ ]:
train_m['anomaly_clip'] = train_m.groupby('nama_pos', observed=True)['anomaly'].transform(
    lambda s: s.clip(s.quantile(0.01), s.quantile(0.99)))

train_sorted = train_m.sort_values('datetime').reset_index(drop=True)
n_blocks = 5
block_id = pd.qcut(train_sorted.index, n_blocks, labels=False)
train_sorted['pred_A_oof'] = np.nan
for b in range(n_blocks):
    fold_tr = train_sorted[block_id != b]
    fold_va = train_sorted[block_id == b]
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(fold_tr[FEATS_A], fold_tr['anomaly_clip'], categorical_feature=['nama_pos'])
    train_sorted.loc[fold_va.index, 'pred_A_oof'] = m.predict(fold_va[FEATS_A])
train_m = train_sorted

model_A = lgb.LGBMRegressor(**lgb_params)
model_A.fit(train_m[FEATS_A], train_m['anomaly_clip'], categorical_feature=['nama_pos'])
test_m['pred_A'] = model_A.predict(test_m[FEATS_A])
print('Stage A final model selesai di-fit.')

Stage A final model selesai di-fit.


In [ ]:
train_m['upstream_pos'] = train_m['nama_pos'].map(upstream_map)
test_m['upstream_pos'] = test_m['nama_pos'].map(upstream_map)

up_train = train_m[['datetime', 'nama_pos', 'pred_A_oof']].rename(columns={'nama_pos': 'upstream_pos', 'pred_A_oof': 'upstream_pred_anomaly'})
up_test = test_m[['datetime', 'nama_pos', 'pred_A']].rename(columns={'nama_pos': 'upstream_pos', 'pred_A': 'upstream_pred_anomaly'})
train_m = train_m.merge(up_train, on=['datetime', 'upstream_pos'], how='left')
test_m = test_m.merge(up_test, on=['datetime', 'upstream_pos'], how='left')
train_m['upstream_pred_anomaly'] = train_m['upstream_pred_anomaly'].fillna(0)
test_m['upstream_pred_anomaly'] = test_m['upstream_pred_anomaly'].fillna(0)

FEATS_B = FEATS_A + ['upstream_pred_anomaly']
model_B = lgb.LGBMRegressor(**lgb_params)
model_B.fit(train_m[FEATS_B], train_m['anomaly_clip'], categorical_feature=['nama_pos'])
test_m['pred_B'] = model_B.predict(test_m[FEATS_B])
print('Stage B final model selesai di-fit.')

Stage B final model selesai di-fit.


In [ ]:
test_m['pred_anomaly_blend'] = 0.5 * test_m['pred_A'] + 0.5 * test_m['pred_B']
test_m['tma_pred'] = test_m['baseline'] + test_m['pred_anomaly_blend']

hist_range = train_m.groupby('nama_pos', observed=True)['tma_mdpl'].agg(['min', 'max', 'std'])
def clip_row(row):
    lo = hist_range.loc[row['nama_pos'], 'min'] - 2 * hist_range.loc[row['nama_pos'], 'std']
    hi = hist_range.loc[row['nama_pos'], 'max'] + 2 * hist_range.loc[row['nama_pos'], 'std']
    return min(max(row['tma_pred'], lo), hi)

test_m['tma_pred_clipped'] = test_m.apply(clip_row, axis=1)
n_clipped = (test_m['tma_pred_clipped'] != test_m['tma_pred']).sum()
print(f'{n_clipped} dari {len(test_m)} prediksi kena safety clip (di luar rentang historis pos-nya)')

0 dari 21780 prediksi kena safety clip (di luar rentang historis pos-nya)


In [ ]:
submission = test_m[['id', 'tma_pred_clipped']].rename(columns={'tma_pred_clipped': 'tma_mdpl'})
submission = sample_sub[['id']].merge(submission, on='id', how='left')

assert submission.shape[0] == sample_sub.shape[0], 'Jumlah baris submission nggak cocok!'
assert submission['tma_mdpl'].isna().sum() == 0, 'Ada prediksi yang kosong!'

submission.to_csv('/content/submission.csv', index=False)
print('Submission tersimpan di /content/submission.csv')
print(submission.shape)
submission.head(10)

Submission tersimpan di /content/submission.csv
(21780, 2)


,id,tma_mdpl
0,2025-09-19 06:00:00 - Arjowinangun - Pacitan,0.870035
1,2025-09-19 12:00:00 - Arjowinangun - Pacitan,0.867130
2,2025-09-19 18:00:00 - Arjowinangun - Pacitan,0.892096
3,2025-09-20 06:00:00 - Arjowinangun - Pacitan,0.906013
4,2025-09-20 12:00:00 - Arjowinangun - Pacitan,0.909119
5,2025-09-20 18:00:00 - Arjowinangun - Pacitan,0.907659
6,2025-09-21 06:00:00 - Arjowinangun - Pacitan,0.909576
7,2025-09-21 12:00:00 - Arjowinangun - Pacitan,0.911347
8,2025-09-21 18:00:00 - Arjowinangun - Pacitan,0.905332
9,2025-09-22 06:00:00 - Arjowinangun - Pacitan,0.912918


## 8. Sanity Check Terakhir

Cek cepat: apakah rata-rata prediksi per pos di test itu masuk akal dibanding rentang historisnya di train (nggak ada yang meleset jauh).

In [ ]:
check = test_m.groupby('nama_pos', observed=True)['tma_pred_clipped'].agg(['mean', 'min', 'max'])
check.columns = ['pred_mean', 'pred_min', 'pred_max']
hist = train.groupby('nama_pos')['tma_mdpl'].agg(['mean', 'min', 'max'])
hist.columns = ['train_mean', 'train_min', 'train_max']
comparison = check.join(hist)
comparison.round(2)

,pred_mean,pred_min,pred_max,train_mean,train_min,train_max
nama_pos,,,,,,
Arjowinangun - Pacitan,1.28,0.84,2.38,1.12,0.35,4.85
Babat,6.37,5.50,7.49,6.31,4.41,9.66
Badegan,122.50,121.99,123.25,122.40,121.90,123.96
Bengkelolor,10.23,8.28,12.40,9.81,7.59,13.52
Boboh Kali Lamong,4.36,2.82,6.37,4.11,2.10,7.60
Bojonegoro - Kali Kethek,9.68,7.16,12.08,9.06,0.00,225.13
Brangkal,13.17,12.43,14.33,12.95,11.69,19.04
Cepu,18.08,16.50,20.19,17.53,15.75,23.67
Colo Weir,107.76,106.22,108.79,107.79,102.19,109.70


## 9. Ringkasan & Next Steps

**Yang udah dikerjain:**
- Target anomali (deviasi dari baseline musiman Fourier per pos) nyelesain masalah skala TMA yang njomplang antar pos.
- Fitur eksogen dengan rolling window (24h/72h/168h) buat curah hujan & soil moisture, sesuai temuan EDA kalau versi akumulasi lebih nyambung ke TMA daripada versi instan.
- Fitur upstream (Stage A → Stage B cascade) buat manfaatin korelasi antar pos.
- Direct forecasting murni dari fitur eksogen (nggak pakai lag TMA sendiri), biar generalize ke horizon berapa aja penting karena test-nya nyampe 8 bulan ke depan.
- Objective `regression_l1` + winsorizing target, biar robust ke outlier spike banjir.
- Validasi jujur pakai walk-forward holdout (bukan random split, dan baseline di-refit biar nggak bocor).
- Safety clip biar prediksi nggak ekstrapolasi liar di luar rentang historis.

**Next steps kalau mau push lebih jauh** (dari diskusi EDA kemarin):
- Coba model deep learning multi-series (TFT/N-HiTS/PatchTST via `darts` atau `neuralforecast`) yang belajar pola lintas-pos secara otomatis.
- Hybrid physics+ML: bikin baseline hidrologi konseptual sederhana, ML belajar residualnya aja.
- Tuning ulang peta upstream beberapa pasangan (terutama yang ngelibatin Gunungsari, yang datanya paling bolong) korelasinya mungkin nggak reliable karena overlap datanya dikit.
- Eksperimen quantile regression / model linear sebagai pelengkap ensemble, buat nge-anchor prediksi di kondisi ekstrapolasi (inget temuan EDA soal distribution shift nino_34).
- Coba threshold winsorizing yang berbeda-beda, atau pisahin modeling buat "kondisi normal" vs "kondisi banjir ekstrem".